# Book Ecosystem — Location Master Table & Dashboard Foundation

This notebook builds an **updatable location-level master table** from the Reading Nation partner statistics and prepares visualizations for:

1. **Book Distribution Dashboard** — books disseminated, children served, LFL boxes, circulation
2. **Ecosystem Density Map** (future) — GIS mapping and book-flow network views

> **Data note:** Numbers come from `Combined Statistics_All Partners.xlsx` (converted to CSV).  
> **LFL box counts, verified coordinates, Kenya, and Ukraine** are not in the source file yet — update `data/location_registry.csv` with Sara & Gaby.

## Setup

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path(".")
DATASET = ROOT / "dataset"
REGISTRY = ROOT / "data" / "location_registry.csv"
OUTPUT = DATASET / "location_master.csv"

LOCATION_ALIASES = {
    "EBCI: Qualla Boundary Head Start": "ebci_qualla_hs",
    "EBCI: Cherokee Elementary School": "ebci_cherokee_elem",
    "EBCI: Snowbird Community Library": "ebci_snowbird_lib",
    "Lumbee: Pembroke Head Start": "lumbee_pembroke_hs",
    "Lumbee: Pembroke Elementary School": "lumbee_pembroke_elem",
    "Lumbee: Robeson Public Library": "lumbee_robeson_lib",
    "Lumbee: Pembroke Boys & Girls Club": "lumbee_robeson_lib",
    "NC: Lame Deer Head Start": "nc_lame_deer_hs",
    "NC: Lame Deer Elementary School": "nc_lame_deer_elem",
    "NC: Woodenlegs Library": "nc_woodenlegs_lib",
    "SDP: Santo Domingo Pueblo Head Start": "sdp_santo_hs",
    "SDP: Santo Domingo Elementary School": "sdp_santo_elem",
    "SDP: Santo Domingo Pueblo Public Library": "sdp_santo_pub_lib",
    "Yurok: Ke'pel Head Start": "yurok_kepel_hs",
    "Yurok: O Me-nok Learning Center": "yurok_omenok",
    "Yurok: 'O Me-nok  Learning Center": "yurok_omenok",
    "Yurok: Del Norte Public Library": "yurok_del_norte_lib",
    "Yurok: Del Norte County Public Library?": "yurok_del_norte_lib",
    "Yurok: Boys & Girls Club LFL": "yurok_bgc_lfl",
    "Yurok: 3 Yurok Tribe Head Start Sites": "yurok_kepel_hs",
}

ECOSYSTEM_TYPE_FROM_CATEGORY = {
    "Head Start LFL": "Preschool/Head Start",
    "Elementary School LFL": "Elementary School",
    "Public/Tribal Library or Community LFL": "Public/Community Library",
}


def parse_numeric(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip().replace(",", "")
    match = re.search(r"[\d.]+", text)
    return float(match.group()) if match else None


def load_registry():
    registry = pd.read_csv(REGISTRY)
    registry["lfl_boxes"] = pd.to_numeric(registry["lfl_boxes"], errors="coerce")
    return registry


def load_dissemination():
    df = pd.read_csv(DATASET / "dissemination.csv")
    df = df[df["location"] != "TOTAL"].copy()
    df["location_key"] = df["location"].map(LOCATION_ALIASES)
    return df


def load_enrollment_latest():
    df = pd.read_csv(DATASET / "enrollment.csv")
    df["location_key"] = df["location"].map(LOCATION_ALIASES)
    df = df.sort_values("period").groupby("location_key", as_index=False).tail(1)
    return df.rename(columns={"enrollment_count": "children_served", "period": "enrollment_period"})


def load_library_usage_latest():
    df = pd.read_csv(DATASET / "library_usage.csv")
    df["location_key"] = df["location"].map(LOCATION_ALIASES)
    df = df.sort_values("period").groupby("location_key", as_index=False).tail(1)
    return df.rename(
        columns={
            "period": "circulation_period",
            "circulation": "circulation_raw",
            "program_attendance": "program_attendance",
            "visitation": "visitation_raw",
        }
    )


def load_reading_log_totals():
    df = pd.read_csv(DATASET / "reading_logs.csv")
    df["location_key"] = df["location"].map(LOCATION_ALIASES)
    return df.groupby("location_key", as_index=False).agg(
        books_read_total=("books_read", lambda s: pd.to_numeric(s, errors="coerce").sum()),
        participants_met_minimum=("met_minimum", lambda s: pd.to_numeric(s, errors="coerce").sum()),
    )


def summarize_gaps(row):
    gaps = []
    if pd.isna(row.get("children_served")):
        gaps.append("children_served")
    if pd.isna(row.get("books_distributed")):
        gaps.append("books_distributed")
    if pd.isna(row.get("lfl_boxes")):
        gaps.append("lfl_boxes")
    if pd.isna(row.get("latitude")) or pd.isna(row.get("longitude")):
        gaps.append("coordinates")
    if pd.isna(row.get("circulation_numeric")) and pd.isna(row.get("books_read_total")):
        gaps.append("circulation_or_reading_log")
    return "; ".join(gaps)


def build_master_table(save=True):
    registry = load_registry()
    dissemination = load_dissemination()
    enrollment = load_enrollment_latest()
    library = load_library_usage_latest()
    reading_logs = load_reading_log_totals()

    books = dissemination.groupby("location_key", as_index=False).agg(
        books_distributed=("books_distributed", "sum"),
        dissemination_categories=("category", lambda s: "; ".join(sorted(set(s)))),
    )

    master = registry.merge(books, on="location_key", how="left")
    master = master.merge(enrollment[["location_key", "children_served", "enrollment_period"]], on="location_key", how="left")
    master = master.merge(
        library[["location_key", "circulation_period", "circulation_raw", "program_attendance", "visitation_raw"]],
        on="location_key",
        how="left",
    )
    master = master.merge(reading_logs, on="location_key", how="left")

    master["circulation_numeric"] = master["circulation_raw"].map(parse_numeric)
    master["visitation_numeric"] = master["visitation_raw"].map(parse_numeric)

    ecosystem_total = master["books_distributed"].sum(skipna=True)
    master["pct_of_ecosystem_books"] = master["books_distributed"].apply(
        lambda x: round(100 * x / ecosystem_total, 1) if pd.notna(x) and ecosystem_total else None
    )
    master["data_gaps"] = master.apply(summarize_gaps, axis=1)

    column_order = [
        "location_key", "location_name", "partner", "ecosystem_type",
        "children_served", "enrollment_period", "books_distributed", "pct_of_ecosystem_books",
        "lfl_boxes", "books_read_total", "participants_met_minimum",
        "circulation_numeric", "circulation_period", "circulation_raw",
        "program_attendance", "visitation_numeric", "visitation_raw",
        "latitude", "longitude", "coordinates_verified",
        "dissemination_categories", "notes", "data_gaps",
    ]
    master = master[column_order].sort_values(["partner", "ecosystem_type", "location_name"])

    if save:
        OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        master.to_csv(OUTPUT, index=False)

    return master


def partner_summary(master):
    summary = master.groupby("partner", as_index=False).agg(
        locations=("location_key", "count"),
        children_served=("children_served", "sum"),
        books_distributed=("books_distributed", "sum"),
        lfl_boxes=("lfl_boxes", "sum"),
        books_read_total=("books_read_total", "sum"),
        circulation=("circulation_numeric", "sum"),
    )
    total_books = summary["books_distributed"].sum()
    summary["pct_of_ecosystem_books"] = (100 * summary["books_distributed"] / total_books).round(1)
    return summary


def ecosystem_totals_row(master):
    return pd.Series(
        {
            "location_name": "Total — All Book Ecosystems",
            "partner": "ALL",
            "ecosystem_type": "Book Ecosystem",
            "children_served": master["children_served"].sum(skipna=True),
            "books_distributed": master["books_distributed"].sum(skipna=True),
            "pct_of_ecosystem_books": 100.0,
            "lfl_boxes": master["lfl_boxes"].sum(skipna=True),
            "books_read_total": master["books_read_total"].sum(skipna=True),
            "circulation_numeric": master["circulation_numeric"].sum(skipna=True),
        }
    )


pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

## 1. Source tables (relationship overview)

All tables link through **`partner`** and **`location_key`** (canonical site ID).  
There are no database foreign keys — we merge manually using `data/location_registry.csv` and the alias mappings defined in the Setup cell above.

In [ ]:
source_tables = {
    "dissemination": pd.read_csv(DATASET / "dissemination.csv"),
    "enrollment": pd.read_csv(DATASET / "enrollment.csv"),
    "library_usage": pd.read_csv(DATASET / "library_usage.csv"),
    "reading_logs": pd.read_csv(DATASET / "reading_logs.csv"),
    "reading_assessments": pd.read_csv(DATASET / "reading_assessments.csv"),
    "reading_scores": pd.read_csv(DATASET / "reading_scores.csv"),
    "location_registry": pd.read_csv(REGISTRY),
}

overview = pd.DataFrame(
    {
        "table": source_tables.keys(),
        "rows": [len(df) for df in source_tables.values()],
        "role": [
            "Books distributed by site (LFL programs)",
            "Head Start & school enrollment by year",
            "Library circulation, programs, visits",
            "Monthly reading-log book counts",
            "State/report-card reading metrics",
            "K-readiness & 4th grade snapshots",
            "Editable site metadata (coords, LFL boxes)",
        ],
    }
)
overview

## 2. Build the location master table

Each row = one physical/program location. Columns match the dashboard spec:

In [ ]:
master = build_master_table(save=True)
print(f"Saved: {DATASET / 'location_master.csv'}  ({len(master)} locations)")

dashboard_cols = [
    "location_name",
    "partner",
    "ecosystem_type",
    "children_served",
    "books_distributed",
    "pct_of_ecosystem_books",
    "lfl_boxes",
    "books_read_total",
    "circulation_numeric",
    "latitude",
    "longitude",
    "data_gaps",
]
master[dashboard_cols]

### EBCI example (mirrors the illustrative dashboard layout)

Compare to the sample table in the project brief. **These are real extracted numbers**, not the AI-generated illustration.

In [ ]:
ebci = master[master["partner"] == "EBCI"].copy()

ebci_display = ebci[
    [
        "location_name",
        "ecosystem_type",
        "children_served",
        "books_distributed",
        "pct_of_ecosystem_books",
        "lfl_boxes",
    ]
].rename(
    columns={
        "location_name": "Location",
        "ecosystem_type": "Type",
        "children_served": "Children Served",
        "books_distributed": "Books Disseminated",
        "pct_of_ecosystem_books": "% of Ecosystem Books",
        "lfl_boxes": "Little Free Library Boxes",
    }
)

ebci_total = pd.DataFrame(
    [
        {
            "Location": "Total",
            "Type": "Book Ecosystem",
            "Children Served": ebci["children_served"].sum(skipna=True),
            "Books Disseminated": ebci["books_distributed"].sum(skipna=True),
            "% of Ecosystem Books": round(ebci["pct_of_ecosystem_books"].sum(skipna=True), 1),
            "Little Free Library Boxes": ebci["lfl_boxes"].sum(skipna=True),
        }
    ]
)

pd.concat([ebci_display, ebci_total], ignore_index=True)

## 3. Partner-level summary

In [ ]:
summary = partner_summary(master)
totals = ecosystem_totals_row(master)

summary_display = summary.rename(
    columns={
        "partner": "Partner",
        "locations": "Locations",
        "children_served": "Children Served",
        "books_distributed": "Books Disseminated",
        "lfl_boxes": "LFL Boxes",
        "books_read_total": "Reading Log Books",
        "circulation": "Circulation (numeric)",
        "pct_of_ecosystem_books": "% of All Books",
    }
)
summary_display

## 4. Book Distribution Dashboard — visualization

Bar chart: books disseminated by location (size of each hub).

In [ ]:
plot_df = master.dropna(subset=["books_distributed"]).sort_values("books_distributed", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Books disseminated by location
axes[0].barh(plot_df["location_name"], plot_df["books_distributed"], color="#2E86AB")
axes[0].set_title("Books Disseminated by Location")
axes[0].set_xlabel("Books")

# Children served (where available)
kids = master.dropna(subset=["children_served"]).sort_values("children_served", ascending=True)
axes[1].barh(kids["location_name"], kids["children_served"], color="#A23B72")
axes[1].set_title("Children Served (latest enrollment)")
axes[1].set_xlabel("Children")

plt.tight_layout()
plt.show()

In [ ]:
# Partner share of total books — pie chart
partner_books = summary.set_index("partner")["books_distributed"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    partner_books,
    labels=partner_books.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"],
)
ax.set_title("Share of Books Disseminated by Partner")
plt.show()

## 5. Ecosystem density map — starter (GIS foundation)

Locations with coordinates are plotted on a map.  
Marker **size** = books disseminated.  
Future work: line thickness between sites = book flow volume (network map).

In [ ]:
try:
    import folium
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "-q"])
    import folium

mapped = master.dropna(subset=["latitude", "longitude"]).copy()
center = [mapped["latitude"].mean(), mapped["longitude"].mean()]

m = folium.Map(location=center, zoom_start=4, tiles="CartoDB positron")

for _, row in mapped.iterrows():
    books = row["books_distributed"] if pd.notna(row["books_distributed"]) else 0
    radius = max(6, min(30, books / 400))
    popup = (
        f"<b>{row['location_name']}</b><br>"
        f"Partner: {row['partner']}<br>"
        f"Type: {row['ecosystem_type']}<br>"
        f"Children: {row['children_served']}<br>"
        f"Books: {row['books_distributed']}<br>"
        f"LFL boxes: {row['lfl_boxes']}"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=radius,
        popup=popup,
        color="#264653",
        fill=True,
        fill_opacity=0.7,
    ).add_to(m)

m

### Book-flow network preview (line thickness = volume)

Connects each partner's Head Start → Elementary → Library sites.  
Line weight scales with `books_distributed` at the destination.

In [ ]:
flow_map = folium.Map(location=center, zoom_start=4, tiles="CartoDB positron")

type_order = ["Preschool/Head Start", "Elementary School", "Public/Community Library", "Tribal Library", "Community Program"]

for partner, group in mapped.groupby("partner"):
    ordered = group.copy()
    ordered["sort_key"] = ordered["ecosystem_type"].apply(
        lambda t: type_order.index(t) if t in type_order else 99
    )
    ordered = ordered.sort_values("sort_key")
    sites = ordered[["latitude", "longitude", "location_name", "books_distributed"]].to_records(index=False)

    for i in range(len(sites) - 1):
        lat1, lon1, name1, _ = sites[i]
        lat2, lon2, name2, books2 = sites[i + 1]
        weight = max(1, (books2 or 0) / 800)
        folium.PolyLine(
            locations=[[lat1, lon1], [lat2, lon2]],
            color="#E76F51",
            weight=weight,
            opacity=0.6,
            tooltip=f"{name1} → {name2}: {books2} books",
        ).add_to(flow_map)

    for _, row in ordered.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=8,
            popup=row["location_name"],
            color="#264653",
            fill=True,
        ).add_to(flow_map)

flow_map

## 6. Data gaps & how to update

| Field | Source today | Action needed |
|-------|--------------|---------------|
| Books disseminated | `dissemination.csv` | Re-run `convert_to_dataset.py` after Excel update |
| Children served | Latest `enrollment.csv` row | Confirm with Sara/Gaby; public libraries may need separate counts |
| LFL boxes | **Missing** | Add to `data/location_registry.csv` |
| Coordinates | Placeholder in registry | Verify with team; set `coordinates_verified=True` |
| Kenya / Ukraine | **Not in workbook** | Add new rows to registry when data arrives |
| Circulation | `library_usage.csv` + reading logs | Some values are text (e.g. "13,171 total / 7,875 youth") |

In [ ]:
gaps = master[master["data_gaps"].astype(str).str.len() > 0][
    ["location_name", "partner", "data_gaps"]
].sort_values("partner")
gaps

## 7. Quick update workflow

1. Edit **`data/location_registry.csv`** (add sites, LFL boxes, verified lat/lon)
2. Update **`Combined Statistics_All Partners.xlsx`** and re-run the CSV conversion cells if needed
3. Re-run this notebook to refresh **`dataset/location_master.csv`**

In [ ]:
# Export a dashboard-ready slim CSV for Tableau / Power BI / GIS tools
export = master[
    [
        "location_key",
        "location_name",
        "partner",
        "ecosystem_type",
        "children_served",
        "books_distributed",
        "pct_of_ecosystem_books",
        "lfl_boxes",
        "books_read_total",
        "circulation_numeric",
        "latitude",
        "longitude",
        "coordinates_verified",
    ]
].copy()

export_path = DATASET / "location_master_dashboard.csv"
export.to_csv(export_path, index=False)
print(f"Dashboard export: {export_path}")
export.head(10)